# Estandarizar los datos

**Sección 3.13 del libro**

Pone las columnas en escalas comparables para que ninguna domine solo por tener números grandes.

> **Basado en:** Karatas, M. (2024). *Developing AI Applications: An Introduction*, cap. 3 «Artificial Neural Networks». Rheinwerk Publishing. Código original del autor, adaptado a Google Colab y comentado en español con fines educativos.


In [ ]:
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

## Cargar los datos

Antes de ejecutar esta celda, sube **adult-2.csv** al panel de archivos de Colab (ícono de carpeta → botón de subir). Si no lo tienes a mano, usa la línea comentada: lee el archivo directamente desde la página de la presentación.

In [ ]:
path = "adult-2.csv"
# path = "https://sinergia-iesa.github.io/introduccion-a-la-creacion-de-redes-neuronales/data/adult-2.csv"   # alternativa sin subir el archivo

data = pd.read_csv(path, delimiter=';')
print(data.head())
print("Columnas con huecos:", data.columns[data.isnull().any()])

Esta vez también quitamos **raza, sexo y país de origen**.

In [ ]:
data.drop(['fnlwgt', 'capital.loss', 'capital.gain', 'race', 'sex', 'native.country'], axis=1, inplace=True)

col = pd.get_dummies(data['income'], dtype=float)
data = data.drop(['income'], axis=1)

conv_num = ['workclass', 'education', 'marital.status', 'occupation', 'relationship']
data[conv_num] = data[conv_num].astype('category')
data[conv_num] = data[conv_num].apply(lambda x: x.cat.codes)

## Escalar

- **StandardScaler** (z): media 0, desviación 1. Para datos tipo campana (edad, ingresos…).
- **MinMaxScaler**: lleva todo a 0…1. Para rangos fijos (horas por semana, píxeles 0–255…).

In [ ]:
s_scaler = StandardScaler()
m_scaler = MinMaxScaler()

cols_to_s_scale = ['age', 'workclass', 'education', 'education.num', 'marital.status', 'occupation', 'relationship']
data[cols_to_s_scale] = s_scaler.fit_transform(data[cols_to_s_scale])

cols_to_m_scale = ['hours.per.week']
data[cols_to_m_scale] = m_scaler.fit_transform(data[cols_to_m_scale])   # el libro dice 'date' (errata)
print(data.describe().round(2))

In [ ]:
train_data, test_data, train_col, test_col = train_test_split(data, col, test_size=0.2, random_state=42)

model = tf.keras.Sequential([
    tf.keras.Input(shape=(data.shape[1],)),   # tantas entradas como columnas
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(2, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_data, train_col, epochs=10)

test_loss, test_acc = model.evaluate(test_data, test_col)
print('Test accuracy:', test_acc)

En producción, el scaler se ajusta **solo con los datos de entrenamiento** (`fit_transform`) y luego se aplica a la prueba con `transform`.